# 📊 Báo cáo Khoa học & Benchmark Pipeline (Milestone 1)

Notebook này phân tích dữ liệu trực tiếp từ hệ thống pipeline thu thập bài báo arXiv theo chuẩn Schema 1.2.

> **Lưu ý quy chuẩn:** Benchmark chính thức chạy trên môi trường **Google Colab CPU** (Linux). Các số liệu chưa đo được biểu diễn là `N/A`, không quy đổi về `0`.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import sys

sys.path.insert(0, 'src')

stats_path = Path('statistics/baseline-001.json')
bench_path = Path('benchmarks/baseline-001.json')

with open(stats_path, 'r', encoding='utf-8') as f:
    stats = json.load(f)
with open(bench_path, 'r', encoding='utf-8') as f:
    bench = json.load(f)

summary = stats.get('summary', {})
paper_sr = summary.get('paper_success_rate')
paper_sr_str = f"{paper_sr:.1%}" if paper_sr is not None else "N/A"
ref_sr = summary.get('reference_metadata_success_rate')
ref_sr_str = f"{ref_sr:.1%}" if ref_sr is not None else "N/A"
mean_refs = summary.get('mean_arxiv_references_per_paper')
mean_refs_str = f"{mean_refs:.1f}" if mean_refs is not None else "N/A"

print(f"Tổng số bài phát hiện: {summary.get('papers_discovered', 0):,}")
print(f"Số bài hoàn thành toàn diện: {summary.get('papers_complete', 0):,} ({paper_sr_str})")
print(f"Trích dẫn arXiv: {summary.get('references_arxiv_total', 0):,} (Tỷ lệ đủ metadata: {ref_sr_str}, TB: {mean_refs_str} ref/bài)")

## 1. Trực quan hóa Tiến độ Thu thập theo Thành viên

In [ ]:
by_owner = stats.get('by_owner', {})
owners = sorted(by_owner.keys())
owner_labels = [o.capitalize() for o in owners]

fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)
discovered = [by_owner[o].get('papers_discovered', 0) for o in owners]
complete = [by_owner[o].get('papers_complete', 0) for o in owners]

x = range(len(owners))
width = 0.35
ax.bar([i - width/2 for i in x], discovered, width, label='Phát hiện', color='#3498db')
ax.bar([i + width/2 for i in x], complete, width, label='Hoàn tất', color='#2ecc71')

ax.set_ylabel('Số lượng bài báo')
ax.set_title('Tiến độ Thu thập & Hoàn thiện theo Thành viên', fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(owner_labels)
ax.legend()
plt.tight_layout()
plt.show()

## 2. Tối ưu hóa Dung lượng Nguồn TeX/BibTeX

In [ ]:
# Dung lượng chỉ tính khi có số đo; hiển thị N/A nếu chưa đo
extracted_raw = [by_owner[o].get('mean_extracted_bytes_before_filter') for o in owners]
retained_raw = [by_owner[o].get('mean_retained_source_bytes') for o in owners]

extracted_kb = [(e / 1024) if e is not None else 0.0 for e in extracted_raw]
retained_kb = [(r / 1024) if r is not None else 0.0 for r in retained_raw]

fig, ax = plt.subplots(figsize=(7, 4), dpi=120)
bars1 = ax.bar([i - 0.15 for i in x], extracted_kb, 0.3, label='Trước lọc (Tổng giải nén)', color='#e74c3c')
bars2 = ax.bar([i + 0.15 for i in x], retained_kb, 0.3, label='Sau lọc (Chỉ TeX/BibTeX)', color='#27ae60')

for i, (e_val, r_val) in enumerate(zip(extracted_raw, retained_raw)):
    if e_val is None:
        ax.text(i, 5, 'N/A', ha='center', va='bottom', fontsize=9, color='#888888', fontweight='bold')

ax.set_ylabel('Dung lượng trung bình (KB)')
ax.set_title('Dung lượng Nguồn Trước vs. Sau khi Lọc File Phụ trợ', fontweight='bold')
ax.set_xticks(list(x))
ax.set_xticklabels(owner_labels)
ax.legend()
plt.tight_layout()
plt.show()

## 3. Hiệu năng & Tài nguyên Hệ thống

In [ ]:
peak_ram = bench.get('peak_ram_bytes')
mean_ram = bench.get('mean_ram_bytes')
peak_disk = bench.get('peak_disk_bytes')
final_output = bench.get('final_output_bytes')
wall_sec = bench.get('wall_seconds')

peak_ram_str = f"{peak_ram / (1024 * 1024):.2f} MB" if peak_ram is not None else "N/A"
mean_ram_str = f"{mean_ram / (1024 * 1024):.2f} MB" if mean_ram is not None else "N/A"
peak_disk_str = f"{peak_disk / 1024:.2f} KB" if peak_disk is not None else "N/A"
final_output_str = f"{final_output / 1024:.2f} KB" if final_output is not None else "N/A"
wall_str = f"{wall_sec:.2f}s" if wall_sec is not None else "N/A"

print(f"Thời gian thực thi toàn lệnh (Wall time): {wall_str}")
print(f"Bộ nhớ RAM cao nhất (Peak RSS): {peak_ram_str}")
print(f"Bộ nhớ RAM trung bình: {mean_ram_str}")
print(f"Phương pháp đo RAM: {bench.get('method', 'N/A')}")
print(f"Dung lượng đầu ra hoàn chỉnh: {final_output_str}")
print(f"Dung lượng đĩa đỉnh (Peak disk): {peak_disk_str}")
print(f"Mã thoát (Exit code): {bench.get('exit_code', 'N/A')}")
print(f"Hệ điều hành: {bench.get('environment', {}).get('os', 'Unknown')}")
print(f"CPU: {bench.get('environment', {}).get('processor', 'Unknown')}")